In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True,
)

### stream()으로 들어오는 각 chunk를 그대로 구분해서 출력해보자

In [2]:
# LLM이 답변을 한꺼번에 주지 않고 조각(chunk) 단위로 보내기 때문에, c에 조각 하나씩 들어옵니다.
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    # c.content가 문자열(String)인지 확인합니다. 문자열이면 바로 출력할 수 있습니다.
    if isinstance(c.content, str):
        # 문자열이면 그대로 출력합니다. end=""는 줄바꿈을 하지 않고 이어 붙이기입니다.
        print(c.content, end="")
    # 문자열이 아니라 리스트 형태라면, 리스트 안의 항목을 하나씩 꺼냅니다.
    else:
        for item in c.content:
            # 리스트 안에는 reasoning, text 같은 여러 종류가 있을 수 있으므로 text만 골라냅니다.
            if item.get("type") == "text":
                # 최종 답변인 text만 출력합니다. |는 각 스트리밍 조각이 어디서 끊겼는지 보기 위한 표시일 뿐이고, 깔끔하게 출력하려면 end=""로 바꾸면 됩니다.
                print(item["text"], end="|")

|잘| 지|냈|어|.| 물|어|봐|줘|서| 고|마|워|.|  
|한국| 사회|의| 문제|는| 하나|로| 단|정|하기| 어렵|지만|,| 서로| 연결|된| 몇| 가지| 큰| 흐|름|으로| 볼| 수| 있어|.

|###| |1|.| 저|출|생|과| 인|구| 고|령|화|
|주|거|비|와| 교육|비|가| 높|고|,| 고|용|이| 불|안|정|하며|,| 돌|봄| 부담|이| 개인|과| 가족|에게| 집중|되|면서| 출|산|을| 선택|하기| 어려|운| 환경|이| 됐|어|.| 동시에| 고|령| 인|구|가| 빠|르게| 늘|어| 연|금|,| 의료|,| 돌|봄| 재|정|에| 부담|이| 커|지고| 있|지|.|  
|단|순|히| “|청|년|들이| 결|혼|·|출|산|을| 기|피|한다|”고| 보기|보다|,| 안정|적인| 일|자리|와| 주|거|,| 성|평|등|한| 돌|봄| 분|담|이| 부족|한| 구조|의| 문제|로| 보는| 게| 중요|해|.

|###| |2|.| 주|거| 불|안|과| 자|산| 격|차|
|특|히| 수도|권|의| 집|값|과| 임|대|료|는| 청|년|과| 서|민|에게| 큰| 부담|이|야|.| 부모|의| 경제|력|이나| 보|유| 자|산|이| 교육|,| 주|거|,| 취|업| 기|회|에| 영향을| 주|면서| “|출|발|선|이| 다|르|다|”|는| 인|식|도| 커|졌|어|.|  
|부|동|산|이| 단|순|한| 거|주| 공간|을| 넘어| 가장| 중요한| 자|산| 증|식| 수|단|이| 되|면서| 세|대|·|계|층| 갈|등|도| 심|해|지고| 있어|.

|###| |3|.| 노동|시|장의| 양|극|화|
|대|기업|과| 중|소|기업|,| 정|규|직|과| 비|정|규|직|,| 수도|권|과| 지방| 사이|의| 격|차|가| 커|.| 취|업| 경쟁|은| 치|열|한|데| 일|자리|의| 안정|성과| 임|금| 수준|은| 크게| 다|르고|,| 노동|시간|과| 직|장| 내| 스트|레스|도| 여|전히| 문제|야|.|  
|플|랫폼| 노동|자|나| 프|리|랜|서|처럼| 기존| 제|도의| 보호|를| 충분|히| 받|지|

In [3]:
import sys
sys.path.append("..")

from langchain_tool_functions import tools, tool_dict

llm_with_tools = llm.bind_tools(tools)

messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:  [] tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:  [{'type': 'function_call', 'name': 'get_current_time', 'arguments': '', 'call_id': 'call_GO2kSOp0FYpaV6NNlbfXCavi', 'id': 'fc_0990e5f4d2240e67006aa2ec12750487d08ff24a51a21af467', 'index': 0}] tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_GO2kSOp0FYpaV6NNlbfXCavi', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:  [{'type': 'function_call', 'name': 'get_current_time', 'arguments': '{"', 'call_id': 'call_GO2kSOp0FYpaV6NNlbfXCavi', 'id': 'fc_0990e5f4d2240e67006aa2ec12750487d08ff24a51a21af467', 'index': 0}] tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_GO2kSOp0FYpaV6NNlbfXCavi', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:  [{'type': 'function_call', 'name': 'get_current_time',

In [4]:
gathered

AIMessageChunk(content=[{'type': 'function_call', 'name': 'get_current_time', 'arguments': '{"timezone":"Asia/Busan","location":"부산"}', 'call_id': 'call_GO2kSOp0FYpaV6NNlbfXCavi', 'id': 'fc_0990e5f4d2240e67006aa2ec12750487d08ff24a51a21af467', 'index': 0}], additional_kwargs={}, response_metadata={'model_provider': 'openai', 'id': 'resp_0990e5f4d2240e67006aa2ec119aa087d082e3dbd4b2e69d3a', 'created_at': 1789062161.0, 'metadata': {}, 'model': 'gpt-5.6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_name': 'gpt-5.6-luna'}, id='resp_0990e5f4d2240e67006aa2ec119aa087d082e3dbd4b2e69d3a', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Busan', 'location': '부산'}, 'id': 'call_GO2kSOp0FYpaV6NNlbfXCavi', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 206, 'output_tokens': 27, 'total_tokens': 233, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}, tool_call_

### 툴이 잘못된 타임존을 인식함 
- {'timezone': 'Asia/Busan', 'location': '부산'}

In [5]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Busan', 'location': '부산'}


UnknownTimeZoneError: 'Asia/Busan'

### 프롬프트를 고쳐서 시도

In [ ]:
messages = [
    # LLM에게 tools를 사용할 수 있다는 것을 알려줌
    # 부산은 한국이므로 올바른 timezone을 사용하도록 추가 정보 제공
    SystemMessage("너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다. 부산은 대한민국에 있으므로 timezone은 Asia/Seoul이다."),

    # 사용자의 질문
    HumanMessage("부산은 지금 몇시야?"),
]

# LLM이 질문을 보고 사용할 tool과 전달할 인자를 결정
gathered = llm_with_tools.invoke(messages)

# LLM의 tool 호출 결과를 대화 기록에 추가
messages.append(gathered)

In [ ]:
# LLM이 요청한 tool들을 하나씩 실행
for tool_call in gathered.tool_calls:
    # tool의 이름을 이용해서 실제 Python 함수를 선택
    selected_tool = tool_dict[tool_call["name"]]

    # LLM이 전달한 인자를 사용하여 tool 실행
    tool_msg = selected_tool.invoke(tool_call)

    # tool 실행 결과를 대화 기록에 추가
    messages.append(tool_msg)

# LLM이 어떤 tool을 호출했는지 확인
print(gathered.tool_calls)

Asia/Seoul (부산) 현재시각 2026-09-11 02:53:58 
[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_N1OwOcV8WfNnMCg5pFpSlbhD', 'type': 'tool_call'}]


In [ ]:
# tool 실행 결과가 포함된 messages를 LLM에 전달하고
# 최종 답변을 chunk 단위로 streaming
for c in llm.stream(messages):
    if isinstance(c.content, list):
        for item in c.content:
            if item.get("type") == "text":
                print(f"|{item['text']}", end="")
    # content가 문자열인 경우
    elif c.content:
        # chunk마다 |를 붙여서 streaming되는 모습을 확인
        print(f"|{c.content}", end="|")

||부|산|은| 지금| **|오|전| |2|시| |53|분|**|입니다|.|

### messages = 대화 기록
```txt
messages
│
├─ HumanMessage      ← 사용자 질문
│
├─ AIMessage         ← LLM이 "이 Tool을 호출해!"라고 요청
│    └─ tool_calls
│
├─ ToolMessage       ← 실제 Tool 실행 결과
│    └─ content
│
└─ AIMessage         ← Tool 결과를 보고 만든 최종 LLM 답변
     └─ content
```

### chunk = Streaming으로 잘라서 들어오는 조각
```txt
llm.stream(messages)

        ↓

chunk → "부산은"
chunk → " 현재"
chunk → " ..."
chunk → "입니다."
```